# 01 · TA-LSTMAE: main results, ablation and uncertainty

Reproduces Tables 2–7 of the revised manuscript under the corrected, leakage-free pipeline (Section 4.8):
* **Main results** (seed 42): Table 4 row, raw confusion matrices, and Tables 5–6 temporal metrics. The **causal (onset-forward) MTTD is the primary metric**; the symmetric ±5-event value is reported only as a sensitivity result (Reviewer 1, comment 1).
* **Repeated independent runs** (5 seeds, mean ± SD and 95% CI) of every ablation configuration and of the causal-only model (R1-6, R1-5, R2-5).
* **MTTD sensitivity** to matching tolerance (0, ±1, ±3, ±5 events) and to the synthetic inter-arrival scale (R1-2).
* Checkpoints, per-run predictions, partition indices, timestamps, class weights, a threshold sweep, ECS alerts, an environment file and a SHA-256 manifest.

The controlled head-to-head comparison with re-implemented baselines is in `02_head_to_head_baselines.ipynb`. The measured end-to-end latency through Elasticsearch/Logstash is in `03_end_to_end_latency.ipynb`.

## 1. Setup, configuration and environment file

In [ ]:
import os, sys, json, warnings
from pathlib import Path
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
ROOT = Path.cwd() if (Path.cwd() / "talstmae").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import tensorflow as tf
from talstmae import Config, set_all_seeds
from talstmae import data as DT, models as M, metrics as MT, experiment as X, alerts as AL

SMOKE_TEST = False      # True = tiny synthetic data, ONLY to check that the code executes. Never report its numbers.
RESUME = True           # continue an interrupted run (finished models/seeds are skipped)
cfg = Config().smoke() if SMOKE_TEST else Config()
if os.environ.get("TALSTMAE_SETTINGS"):          # settings written by 00_RUN_ME.ipynb override the lines above
    _s = json.loads(Path(os.environ["TALSTMAE_SETTINGS"]).read_text())
    _base = Config(**{k: (tuple(v) if k in ("datasets", "seeds") else v) for k, v in _s.get("config", {}).items()})
    SMOKE_TEST, RESUME = bool(_s.get("smoke_test")), bool(_s.get("resume", True))
    cfg = _base.smoke() if SMOKE_TEST else _base
    print("settings from 00_RUN_ME:", _s)
if SMOKE_TEST:
    print("=" * 78 + "\nSMOKE TEST MODE: synthetic placeholder data. Results are INVALID and must not be reported.\n" + "=" * 78)
set_all_seeds(cfg.seed, cfg.deterministic_ops)
print("TensorFlow", tf.__version__, "| outputs ->", cfg.out.resolve())

cfg.save()
import sklearn, keras, platform
env = {"python": sys.version.split()[0], "platform": platform.platform(), "tensorflow": tf.__version__,
       "keras": keras.__version__, "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
       "gpu_devices": [d.name for d in tf.config.list_physical_devices("GPU")]}
(cfg.out / "environment.json").write_text(json.dumps(env, indent=2))
(cfg.out / "environment.txt").write_text("\n".join(f"{k}=={env[k]}" for k in
                                         ("python", "tensorflow", "keras", "numpy", "pandas", "scikit-learn")) + "\n")
env

## 2. Data: strict loading, preprocessing, contiguous splits, timestamps and leakage audit
Loaders never substitute placeholder data; a missing file raises an error. Expected model-input dimensions are 122 / 194 / 78. Note that NSL-KDD has **122** model-input features, so the compression ratio is 122→16 = 86.89% (Reviewer 1, comment 7). UNSW-NB15 has **9 attack categories plus Normal, i.e. 10 classes** (Reviewer 1, comment 8).

In [ ]:
DATA = {ds: DT.prepare_dataset(ds, cfg) for ds in cfg.datasets}
WINDOW_COUNTS = {ds: DT.leakage_audit(DATA[ds], cfg) for ds in cfg.datasets}

## 3. Model configurations and parameter counts (Table 2; Section 4.6)

In [ ]:
display(M.parameter_check(cfg.seq_len))
TABLE2 = {}
for ds, D in DATA.items():
    ae, _ = M.build_tae(D["X_tr"].shape[1], cfg.latent_dim[ds])
    TABLE2[DT.NAMES[ds]] = M.param_table(ae).set_index("layer")["params"]
    with open(cfg.out / ds / "model_summary_TAE.txt", "w") as f:
        ae.summary(print_fn=lambda s, *a, **k: f.write(s + "\n"))
pd.DataFrame(TABLE2)

## 4. Main run (seed 42): TAE + HBLT per dataset
Checkpoints (`tae.keras`, `tae_encoder.keras`, `hblt_full.keras`), test predictions, confusion matrices and metrics are written to `outputs/<dataset>/`. These seed-42 checkpoints are reused by notebook 03.

In [ ]:
RESULTS = {}
for ds, D in DATA.items():
    d = cfg.out / ds
    main_npz = d / "main_seed_predictions.npz"
    if RESUME and main_npz.exists() and (d / "hblt_full.keras").exists():
        _z = np.load(main_npz); p_val, p_te = _z["prob_val"], _z["prob_test"]
        print(f"{ds}: main run already done, loaded saved predictions (resume)")
    else:
        ae, enc, lat = X.train_tae(D, cfg, cfg.seed, save_dir=d)
        clf, h, p_val, p_te = X.train_window_classifier(D, lat, cfg, M.CONFIGS[M.FULL]["branches"], cfg.hblt_epochs[ds], cfg.seed)
        clf.save(d / "hblt_full.keras")
        with open(d / "model_summary_HBLT.txt", "w") as f:
            clf.summary(print_fn=lambda s, *a, **k: f.write(s + "\n"))
        pd.DataFrame(h.history).to_csv(d / "hblt_history.csv", index=False)
        np.savez_compressed(main_npz, prob_val=p_val, prob_test=p_te)
    y, ts = X.test_targets(D, cfg)
    res = MT.evaluate_predictions(y, p_te, ts, cfg.threshold, cfg.match_tolerance, cfg.t_response_s)
    RESULTS[ds] = dict(res=res, p_val=p_val, p_te=p_te)
    yhat = (p_te >= cfg.threshold).astype(np.int8)
    pd.DataFrame({"window_index": np.arange(len(y)), "record_index": np.arange(len(y)) + cfg.seq_len - 1,
                  "timestamp_s": ts, "y_true": y, "y_prob": p_te, "y_pred": yhat}).to_csv(d / "predictions_test.csv", index=False)
    (d / "confusion_matrix.json").write_text(json.dumps({k: res[k] for k in ("TN", "FP", "FN", "TP", "n")}, indent=2))
    X.save_json(res, d / "metrics.json")
    AL.write_bulk_ndjson((AL.ecs_document(ds, i, ts[i], p_te[i], res["MTTD_causal"], cfg.t_response_s)
                          for i in np.flatnonzero(yhat)), d / "alerts_ecs.ndjson")
    print(f"{ds:<11s} Acc={res['accuracy']:.4f} P={res['precision']:.4f} R={res['recall']:.4f} F1={res['f1']:.4f} | "
          f"TN={res['TN']:,} FP={res['FP']:,} FN={res['FN']:,} TP={res['TP']:,} | MTTD causal={res['MTTD_causal']:.4f}s "
          f"(symmetric {res['MTTD_symmetric']:.4f}s) APR={res['APR']:.4f} DR={res['DetectionRate_causal']:.4f}")

## 5. Validation-set decision-threshold sweep (Section 4.8)
The threshold is chosen on validation windows only; the test set is never used for this choice.

In [ ]:
rows = []
for ds, D in DATA.items():
    yv, _ = X.val_targets(D, cfg)
    for thr in np.round(np.arange(0.10, 0.91, 0.05), 2):
        m = MT.classification_metrics(yv, (RESULTS[ds]["p_val"] >= thr).astype(np.int8))
        rows.append(dict(dataset=ds, threshold=thr, **{k: m[k] for k in ("accuracy", "precision", "recall", "f1")}))
SWEEP = pd.DataFrame(rows); SWEEP.to_csv(cfg.out / "threshold_sweep.csv", index=False)
best = SWEEP.loc[SWEEP.groupby("dataset")["f1"].idxmax(), ["dataset", "threshold", "f1"]]
best.merge(SWEEP[SWEEP.threshold == 0.5][["dataset", "f1"]].rename(columns={"f1": "f1_at_0.5"}), on="dataset")

## 6. Repeated independent runs: ablation and causal model with uncertainty (Table 7)
Every configuration is trained with each seed in `cfg.seeds`, with one TAE per (dataset, seed) shared across configurations. The table reports mean ± SD and the 95% t-interval. Paired tests across seeds compare each configuration with the full model, using Holm-adjusted p-values. Small differences whose intervals overlap should not be interpreted as architectural gains.

In [ ]:
ABL = X.run_ablation(DATA, cfg, resume=RESUME)
ABL_SUM = MT.summarize_runs(ABL)
ABL_SUM.to_csv(cfg.out / "ablation_summary.csv", index=False)
display(MT.format_mean_sd(ABL_SUM, ("accuracy", "precision", "recall", "f1", "MTTD_causal")))
tests = [MT.paired_seed_test(ABL, "f1", c, M.FULL, ds) for ds in DATA for c in M.CONFIGS if c != M.FULL]
TESTS = pd.DataFrame(tests)
TESTS["paired_t_p_holm"] = np.nan
ok = TESTS.paired_t_p.notna()
TESTS.loc[ok, "paired_t_p_holm"] = MT.holm(TESTS.loc[ok, "paired_t_p"].values)
TESTS.to_csv(cfg.out / "ablation_paired_tests.csv", index=False); TESTS.round(4)

## 7. MTTD sensitivity: matching tolerance and inter-arrival scale (Reviewer 1, comment 2)
The seed-42 predictions are held fixed. For NSL-KDD and UNSW-NB15 the test timestamps are regenerated with the same seed at several mean inter-arrival times. CICIDS-2018 uses its native timestamps only. **Causal** is the primary rule.

In [ ]:
SENS = []
for ds, D in DATA.items():
    y, _ = X.test_targets(D, cfg)
    yhat = (RESULTS[ds]["p_te"] >= cfg.threshold).astype(np.int8)
    scales = [None] if D["native_timestamps"] else [0.01, 0.05, 0.1, 0.5, 1.0]
    for mean in scales:
        if mean is None:
            ts, label = D["ts_te"][cfg.seq_len - 1:], "native"
        else:
            raw = DT.synthetic_timestamps(len(D["y_te"]), cfg.ts_seeds.get(ds, 2000) + 500, mean)
            ts, label = DT.apply_resolution(raw, cfg.ts_resolution_s[ds])[cfg.seq_len - 1:], f"mean={mean}s"
        for tol in (0, 1, 3, 5):
            for causal in (True, False):
                t = MT.temporal_metrics(y, yhat, ts, tol=tol, causal=causal, t_response=cfg.t_response_s)
                SENS.append(dict(dataset=ds, timestamps=label, tolerance=tol, rule="causal" if causal else "symmetric", **t))
SENS = pd.DataFrame(SENS); SENS.to_csv(cfg.out / "mttd_sensitivity.csv", index=False)
SENS.pivot_table(index=["dataset", "timestamps", "rule"], columns="tolerance", values="MTTD").round(4)

## 8. Regenerated tables and comparison with the manuscript

In [ ]:
TDIR = cfg.out / "tables"; TDIR.mkdir(exist_ok=True)
def save(df, name):
    df.to_csv(TDIR / f"{name}.csv", index=False)
    try: (TDIR / f"{name}.md").write_text(df.to_markdown(index=False))
    except ImportError: pass
    return df
save(pd.DataFrame(TABLE2).reset_index(), "table2_tae_parameters")
save(pd.DataFrame([dict(dataset=DT.NAMES[d], train_records=len(DATA[d]["y_tr"]) + len(DATA[d]["y_val"]),
                        test_records=len(DATA[d]["y_te"]), model_input_dim=DATA[d]["X_tr"].shape[1],
                        train_windows=WINDOW_COUNTS[d][0], val_windows=WINDOW_COUNTS[d][1], test_windows=WINDOW_COUNTS[d][2])
                   for d in DATA]), "table3_dataset_summary")
t4 = save(pd.DataFrame([dict(dataset=DT.NAMES[d], **{k: (round(v, 4) if isinstance(v, float) else v)
                        for k, v in RESULTS[d]["res"].items() if k in ("accuracy", "precision", "recall", "f1", "TN", "FP", "FN", "TP", "n")})
                        for d in DATA]), "table4_ta_lstmae_seed42")
full = ABL_SUM[ABL_SUM.model == M.FULL]
save(MT.format_mean_sd(full, ("accuracy", "precision", "recall", "f1")), "table6_ta_lstmae_mean_sd")
t5 = save(pd.DataFrame([dict(dataset=DT.NAMES[d], MTTD_causal_s=round(RESULTS[d]["res"]["MTTD_causal"], 4),
                             projected_MTTR_causal_s=round(RESULTS[d]["res"]["pMTTR_causal"], 4),
                             MTTD_symmetric_s_sensitivity=round(RESULTS[d]["res"]["MTTD_symmetric"], 4),
                             APR=round(RESULTS[d]["res"]["APR"], 4), DetectionRate=round(RESULTS[d]["res"]["DetectionRate_causal"], 4))
                        for d in DATA]), "table5_temporal_metrics_causal_primary")
save(ABL_SUM.round(4), "table6_ablation_mean_sd_ci")
REPORTED = {"nsl_kdd": dict(accuracy=0.7692, precision=0.9100, recall=0.6599, f1=0.7650, TN=8870, FP=837, FN=4363, TP=8465,
                            MTTD_causal=0.0667, MTTD_symmetric=0.0041, APR=0.9100),
            "unsw_nb15": dict(accuracy=0.8584, precision=0.8294, recall=0.9353, f1=0.8791, TN=28268, FP=8723, FN=2935, TP=42397,
                              MTTD_causal=0.0076, MTTD_symmetric=0.0000, APR=0.8294),
            "cicids2018": dict(accuracy=0.9994, precision=0.9985, recall=1.0000, f1=0.9993, TN=30127, FP=33, FN=0, TP=22259,
                               MTTD_causal=0.0000, MTTD_symmetric=0.0000, APR=0.9985)}
if not cfg.smoke_test:
    cmp = [dict(dataset=DT.NAMES[d], metric=k, reported=v, regenerated=round(float(RESULTS[d]["res"][k]), 4),
                difference=round(float(RESULTS[d]["res"][k]) - v, 4)) for d in DATA for k, v in REPORTED[d].items()]
    display(save(pd.DataFrame(cmp), "comparison_with_manuscript"))
display(t4); display(t5)

## 9. Output manifest (SHA-256 of inputs and outputs)

In [ ]:
import datetime as dt
manifest = {"generated_utc": dt.datetime.now(dt.timezone.utc).isoformat(), "smoke_test": cfg.smoke_test,
            "inputs": DT.PROVENANCE, "environment": env,
            "outputs": {str(p.relative_to(cfg.out)): DT.sha256(p) for p in sorted(cfg.out.rglob("*"))
                        if p.is_file() and p.name != "MANIFEST.json"}}
(cfg.out / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))
print(len(manifest["outputs"]), "files recorded in", cfg.out / "MANIFEST.json")